# AI_LEARNING_ROADMAP_GENERATOR

In [21]:
import math
import re

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPClassifier


# ==========================================================
# 1. LEARNING SKILLS AND TOPICS
# ==========================================================

SKILLS = {

    "Python": {
        "duration": 2,
        "topics": [
            "Basics",
            "Variables",
            "Conditions",
            "Functions",
            "Lists and Dictionaries",
            "OOP Basics"
        ]
    },

    "NumPy": {
        "duration": 1,
        "topics": [
            "Arrays",
            "Indexing",
            "Array Operations",
            "Mathematical Functions"
        ]
    },

    "Pandas": {
        "duration": 1,
        "topics": [
            "DataFrames",
            "Data Cleaning",
            "Filtering",
            "GroupBy",
            "Merge"
        ]
    },

    "SQL": {
        "duration": 2,
        "topics": [
            "SELECT",
            "WHERE",
            "GROUP BY",
            "JOINS",
            "Subqueries"
        ]
    },

    "Statistics": {
        "duration": 2,
        "topics": [
            "Mean and Median",
            "Probability",
            "Standard Deviation",
            "Hypothesis Testing",
            "Correlation"
        ]
    },

    "AI Basics": {
        "duration": 1,
        "topics": [
            "What is AI?",
            "AI vs ML vs Deep Learning",
            "Types of AI",
            "AI Applications",
            "AI Ethics Basics"
        ]
    },

    "Machine Learning": {
        "duration": 3,
        "topics": [
            "Supervised Learning",
            "Unsupervised Learning",
            "Regression",
            "Classification",
            "Train-Test Split",
            "Model Evaluation"
        ]
    },

    "Deep Learning Basics": {
        "duration": 3,
        "topics": [
            "Neurons and Neural Networks",
            "Weights and Bias",
            "Activation Functions",
            "ReLU",
            "Forward Propagation",
            "Loss and Gradient Descent",
            "Overfitting"
        ]
    },

    "NLP Basics": {
        "duration": 3,
        "topics": [
            "Text Cleaning",
            "Tokenization",
            "Stopwords",
            "Stemming and Lemmatization",
            "Bag of Words",
            "TF-IDF",
            "Text Classification"
        ]
    }
}


SKILL_LIST = list(SKILLS.keys())


# ==========================================================
# 2. TARGET ROLES
# ==========================================================

ROLES = {

    "Data Analyst": [
        "Python",
        "SQL",
        "Statistics",
        "Pandas"
    ],

    "Data Scientist": [
        "Python",
        "NumPy",
        "Pandas",
        "SQL",
        "Statistics",
        "AI Basics",
        "Machine Learning",
        "Deep Learning Basics"
    ],

    "ML Engineer": [
        "Python",
        "NumPy",
        "Pandas",
        "AI Basics",
        "Machine Learning",
        "Deep Learning Basics"
    ],

    "NLP Engineer": [
        "Python",
        "Pandas",
        "AI Basics",
        "Machine Learning",
        "Deep Learning Basics",
        "NLP Basics"
    ]
}


# ==========================================================
# 3. SKILL SYNONYMS
# ==========================================================

SKILL_SYNONYMS = {

    "python": "Python",
    "py": "Python",

    "numpy": "NumPy",

    "pandas": "Pandas",

    "sql": "SQL",
    "mysql": "SQL",
    "postgresql": "SQL",

    "statistics": "Statistics",
    "stats": "Statistics",

    "ai": "AI Basics",
    "artificial intelligence": "AI Basics",

    "machine learning": "Machine Learning",
    "machinelearning": "Machine Learning",
    "ml": "Machine Learning",
    "sklearn": "Machine Learning",
    "scikit learn": "Machine Learning",

    "deep learning": "Deep Learning Basics",
    "deeplearning": "Deep Learning Basics",
    "dl": "Deep Learning Basics",
    "neural network": "Deep Learning Basics",
    "neural networks": "Deep Learning Basics",

    "nlp": "NLP Basics",
    "natural language processing": "NLP Basics"
}


# ==========================================================
# 4. TEXT CLEANING
# ==========================================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ==========================================================
# 5. TOKENIZATION
# ==========================================================

def tokenize(text):

    cleaned = clean_text(text)

    return cleaned.split()


# ==========================================================
# 6. SKILL EXTRACTION
# ==========================================================

def extract_skills(text):

    cleaned = clean_text(text)

    detected_skills = []

    for keyword, canonical_skill in SKILL_SYNONYMS.items():

        pattern = r"\b" + re.escape(keyword) + r"\b"

        if re.search(pattern, cleaned):

            if canonical_skill not in detected_skills:

                detected_skills.append(
                    canonical_skill
                )

    # Keep skills in the same order as SKILLS
    ordered_skills = [

        skill
        for skill in SKILL_LIST
        if skill in detected_skills
    ]

    return ordered_skills


# ==========================================================
# 7. ROLE DESCRIPTIONS FOR NLP
# ==========================================================

ROLE_DESCRIPTIONS = {

    "Data Analyst":
        """
        Data Analyst Python SQL Statistics Pandas
        data analysis reports dashboards business insights
        """,

    "Data Scientist":
        """
        Data Scientist Python NumPy Pandas SQL Statistics
        Artificial Intelligence Machine Learning Deep Learning
        prediction models data science
        """,

    "ML Engineer":
        """
        ML Engineer Python NumPy Pandas Artificial Intelligence
        Machine Learning Deep Learning Neural Networks
        model development deployment
        """,

    "NLP Engineer":
        """
        NLP Engineer Python Pandas Artificial Intelligence
        Machine Learning Deep Learning Natural Language Processing
        text data NLP
        """
}


# ==========================================================
# 8. NLP ROLE SUGGESTION
# ==========================================================

def suggest_role(user_goal):

    goal = clean_text(user_goal)

    if not goal:

        return "Data Analyst", 0.0

    # Exact role matching first
    exact_role_matches = {

        "data analyst": "Data Analyst",
        "data scientist": "Data Scientist",
        "ml engineer": "ML Engineer",
        "machine learning engineer": "ML Engineer",
        "nlp engineer": "NLP Engineer"
    }

    for role_text, role_name in exact_role_matches.items():

        if role_text in goal:

            return role_name, 1.0

    # TF-IDF + Cosine Similarity
    documents = []

    for role, description in ROLE_DESCRIPTIONS.items():

        documents.append(
            clean_text(
                role + " " + description
            )
        )

    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(
        documents + [goal]
    )

    similarity_scores = cosine_similarity(
        tfidf_matrix[-1],
        tfidf_matrix[:-1]
    )[0]

    best_index = int(
        np.argmax(similarity_scores)
    )

    role_names = list(
        ROLE_DESCRIPTIONS.keys()
    )

    best_role = role_names[best_index]

    best_score = float(
        similarity_scores[best_index]
    )

    return best_role, best_score


# ==========================================================
# 9. FIND SKILL GAP
# ==========================================================

def find_skill_gap(
    current_skills,
    target_role
):

    required_skills = ROLES[target_role]

    skill_gap = [

        skill

        for skill in required_skills

        if skill not in current_skills
    ]

    return skill_gap


# ==========================================================
# 10. CALCULATE TOTAL DURATION
# ==========================================================

def calculate_duration(
    skill_gap,
    hours_per_day
):

    total_base_weeks = sum(

        SKILLS[skill]["duration"]

        for skill in skill_gap
    )

    total_weeks = math.ceil(

        (total_base_weeks * 2)
        / hours_per_day
    )

    return total_weeks


# ==========================================================
# 11. SAMPLE TRAINING DATA
# ==========================================================

training_data = pd.DataFrame({

    "Python": [
        0, 1, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 0
    ],

    "SQL": [
        0, 0, 1, 1, 1, 1,
        1, 1, 0, 1, 0, 1
    ],

    "Statistics": [
        0, 0, 0, 1, 1, 1,
        1, 0, 1, 1, 1, 0
    ],

    "Machine_Learning": [
        0, 0, 0, 0, 1, 1,
        0, 1, 1, 1, 0, 1
    ],

    "Deep_Learning": [
        0, 0, 0, 0, 0, 1,
        0, 0, 1, 1, 1, 1
    ],

    "Recommendation": [
        "Focus on Core Fundamentals",
        "Focus on Core Fundamentals",
        "Focus on Core Fundamentals",
        "Focus on Core Fundamentals",
        "Continue with Advanced Learning",
        "Continue with Advanced Learning",
        "Focus on Core Fundamentals",
        "Continue with Advanced Learning",
        "Continue with Advanced Learning",
        "Continue with Advanced Learning",
        "Continue with Advanced Learning",
        "Continue with Advanced Learning"
    ]
})


# ==========================================================
# 12. TRAIN NEURAL NETWORK
# ==========================================================

X = training_data[[
    "Python",
    "SQL",
    "Statistics",
    "Machine_Learning",
    "Deep_Learning"
]]

y = training_data["Recommendation"]


model = MLPClassifier(

    hidden_layer_sizes=(5,),

    activation="relu",

    solver="lbfgs",

    max_iter=1000,

    random_state=42
)


model.fit(X, y)


# ==========================================================
# 13. GET AI RECOMMENDATION
# ==========================================================

def get_recommendation(
    current_skills
):

    input_row = {

        "Python":
            int("Python" in current_skills),

        "SQL":
            int("SQL" in current_skills),

        "Statistics":
            int("Statistics" in current_skills),

        "Machine_Learning":
            int(
                "Machine Learning"
                in current_skills
            ),

        "Deep_Learning":
            int(
                "Deep Learning Basics"
                in current_skills
            )
    }


    # Keep feature names exactly same
    input_data = pd.DataFrame(

        [input_row],

        columns=[
            "Python",
            "SQL",
            "Statistics",
            "Machine_Learning",
            "Deep_Learning"
        ]
    )


    prediction = model.predict(
        input_data
    )[0]


    probabilities = model.predict_proba(
        input_data
    )[0]


    confidence = float(
        np.max(probabilities)
    )


    return prediction, confidence


# ==========================================================
# 14. GENERATE PERSONALIZED ROADMAP
# ==========================================================

def generate_roadmap(
    name,
    target_role,
    skills_text,
    hours_per_day,
    goal=""
):

    print("\n" + "=" * 60)

    print(
        "       AI LEARNING ROADMAP GENERATOR"
    )

    print("=" * 60)


    print(
        f"\nHello {name}!"
    )


    # ------------------------------------------------------
    # PROFILE
    # ------------------------------------------------------

    current_skills = extract_skills(
        skills_text
    )


    print("\n1. PROFILE")

    print("-" * 40)


    print(
        "Target Role:",
        target_role
    )


    if current_skills:

        print(
            "Detected Skills:",
            ", ".join(current_skills)
        )

    else:

        print(
            "Detected Skills: None"
        )


    # ------------------------------------------------------
    # NLP ROLE SUGGESTION
    # ------------------------------------------------------

    if goal.strip():

        suggested_role, score = suggest_role(
            goal
        )


        print(
            "\n2. NLP ROLE SUGGESTION"
        )

        print("-" * 40)


        print(
            "Suggested Role:",
            suggested_role
        )


        print(
            "Similarity:",
            f"{score:.2%}"
        )


    # ------------------------------------------------------
    # SKILL GAP
    # ------------------------------------------------------

    section_number = (
        3 if goal.strip() else 2
    )


    print(
        f"\n{section_number}. SKILL GAP"
    )

    print("-" * 40)


    skill_gap = find_skill_gap(
        current_skills,
        target_role
    )


    if skill_gap:

        for skill in skill_gap:

            print(
                f"• {skill}"
            )

    else:

        print(
            "No major skill gap found."
        )


    # ------------------------------------------------------
    # AI RECOMMENDATION
    # ------------------------------------------------------

    section_number += 1


    print(
        f"\n{section_number}. AI RECOMMENDATION"
    )

    print("-" * 40)


    recommendation, confidence = get_recommendation(
        current_skills
    )


    print(
        "Recommendation:",
        recommendation
    )


    print(
        "Recommendation Probability:",
        f"{confidence:.2%}"
    )


    # ------------------------------------------------------
    # PERSONALIZED ROADMAP
    # ------------------------------------------------------

    section_number += 1


    print(
        f"\n{section_number}. PERSONALIZED ROADMAP"
    )

    print("-" * 40)


    if not skill_gap:

        print(
            "You already have the required skills."
        )

        print(
            "You can focus on advanced topics and projects."
        )

        return pd.DataFrame()


    total_weeks = calculate_duration(
        skill_gap,
        hours_per_day
    )


    print(
        f"Estimated Duration: {total_weeks} week(s)"
    )


    print(
        f"Study Time: {hours_per_day:.1f} hour(s) per day"
    )


    roadmap_rows = []


    for step, skill in enumerate(
        skill_gap,
        start=1
    ):

        base_duration = SKILLS[skill]["duration"]


        estimated_duration = round(

            (base_duration * 2)
            / hours_per_day,

            1
        )


        print(
            f"\nStep {step}: {skill}"
        )


        print(
            "Estimated Study Duration:",
            f"{estimated_duration} week(s)"
        )


        print(
            "Topics:"
        )


        for topic in SKILLS[skill]["topics"]:

            print(
                f"   • {topic}"
            )


        roadmap_rows.append({

            "Step":
                step,

            "Skill":
                skill,

            "Topic":
                ", ".join(
                    SKILLS[skill]["topics"]
                ),

            "Duration_Weeks":
                estimated_duration
        })


    roadmap_df = pd.DataFrame(
        roadmap_rows
    )


    return roadmap_df


# ==========================================================
# 15. USER INPUT
# ==========================================================

print("\n" + "=" * 60)

print(
    "       WELCOME TO AI LEARNING ROADMAP GENERATOR"
)

print("=" * 60)


# ----------------------------------------------------------
# NAME
# ----------------------------------------------------------

name = input(
    "\nEnter your name: "
).strip()


if not name:

    name = "User"


# ----------------------------------------------------------
# CURRENT SKILLS
# ----------------------------------------------------------

skills_text = input(

    "\nEnter your current skills "
    "(Example: Python, SQL, Pandas, NumPy): "

).strip()


# ----------------------------------------------------------
# TARGET ROLE KNOWLEDGE
# ----------------------------------------------------------

print(
    "\nDo you know your target role?"
)

print(
    "1. Yes"
)

print(
    "2. No"
)


while True:

    role_known = input(
        "\nSelect an option (1-2 or Yes/No): "
    ).strip().lower()


    if role_known in [
        "1",
        "yes",
        "y"
    ]:

        role_known = "1"

        break


    elif role_known in [
        "2",
        "no",
        "n"
    ]:

        role_known = "2"

        break


    else:

        print(
            "Please enter 1, 2, Yes, or No."
        )


goal = ""


# ==========================================================
# OPTION 1 - USER KNOWS TARGET ROLE
# ==========================================================

if role_known == "1":

    print(
        "\nAvailable Roles:"
    )


    role_list = list(
        ROLES.keys()
    )


    for number, role in enumerate(
        role_list,
        start=1
    ):

        print(
            f"{number}. {role}"
        )


    while True:

        role_choice = input(
            "\nSelect your target role (1-4): "
        ).strip()


        if role_choice.isdigit():

            role_number = int(
                role_choice
            )


            if 1 <= role_number <= len(role_list):

                target_role = role_list[
                    role_number - 1
                ]

                break


        print(
            "Please enter a valid number from 1 to 4."
        )


# ==========================================================
# OPTION 2 - USER DOES NOT KNOW TARGET ROLE
# ==========================================================

else:

    goal = input(

        "\nEnter your career goal: "

    ).strip()


    while not goal:

        print(
            "Career goal cannot be empty."
        )

        goal = input(
            "Enter your career goal: "
        ).strip()


    target_role, role_score = suggest_role(
        goal
    )


    print(
        "\nBased on your career goal,"
    )


    print(
        "NLP Suggested Role:",
        target_role
    )


    print(
        "Similarity:",
        f"{role_score:.2%}"
    )


# ==========================================================
# STUDY HOURS
# ==========================================================

while True:

    hours_input = input(

        "\nHow many hours can you study per day? "

    ).strip()


    try:

        hours_per_day = float(
            hours_input
        )


        if hours_per_day > 0:

            break


        print(
            "Hours must be greater than 0."
        )


    except ValueError:

        print(
            "Please enter a valid number."
        )


# ==========================================================
# GENERATE ROADMAP
# ==========================================================

roadmap = generate_roadmap(

    name=name,

    target_role=target_role,

    skills_text=skills_text,

    hours_per_day=hours_per_day,

    goal=goal
)


# ==========================================================
# SAVE ROADMAP TO CSV
# ==========================================================

if not roadmap.empty:

    roadmap.to_csv(

        "my_ai_learning_roadmap.csv",

        index=False
    )


    print(
        "\nRoadmap saved successfully!"
    )

else:

    print(
        "\nNo new roadmap was required."
    )


       WELCOME TO AI LEARNING ROADMAP GENERATOR



Enter your name:  durga

Enter your current skills (Example: Python, SQL, Pandas, NumPy):  python,pandas



Do you know your target role?
1. Yes
2. No



Select an option (1-2 or Yes/No):  1



Available Roles:
1. Data Analyst
2. Data Scientist
3. ML Engineer
4. NLP Engineer



Select your target role (1-4):  2

How many hours can you study per day?  4



       AI LEARNING ROADMAP GENERATOR

Hello durga!

1. PROFILE
----------------------------------------
Target Role: Data Scientist
Detected Skills: Python, Pandas

2. SKILL GAP
----------------------------------------
• NumPy
• SQL
• Statistics
• AI Basics
• Machine Learning
• Deep Learning Basics

3. AI RECOMMENDATION
----------------------------------------
Recommendation: Focus on Core Fundamentals
Recommendation Probability: 99.99%

4. PERSONALIZED ROADMAP
----------------------------------------
Estimated Duration: 6 week(s)
Study Time: 4.0 hour(s) per day

Step 1: NumPy
Estimated Study Duration: 0.5 week(s)
Topics:
   • Arrays
   • Indexing
   • Array Operations
   • Mathematical Functions

Step 2: SQL
Estimated Study Duration: 1.0 week(s)
Topics:
   • SELECT
   • WHERE
   • GROUP BY
   • JOINS
   • Subqueries

Step 3: Statistics
Estimated Study Duration: 1.0 week(s)
Topics:
   • Mean and Median
   • Probability
   • Standard Deviation
   • Hypothesis Testing
   • Correlation

